# 非線形バネをiLQRで制御する

iLQRの実装を理解するため、単純な非線形モデルである、非線形バネを対象とする。

iLQRでは計算に以下の項目が必要になる。

- forward pass による基準軌道と候補軌道の計算
    - 非線形動力学モデルの離散状態方程式 $X_{n+1} = f(X_n, u_n)$
- 基準軌道の更新と最適軌道の選定
    - 総コスト関数 $J(X, u) = \phi(X_N) + \sum \ell(X_n, u_n)$
- backward pass による入力修正則の係数計算
    - 非線形動力学モデルの一次近似式 $\delta X_{n+1} \approx A_n \delta X_n + B_n \delta u_n$ の $A_n,B_n$
    - 総コスト関数の各項の二次近似式の係数
        - ステージコスト $\ell_{X,n},\ell_{u,n}, \ell_{XX,n}, \ell_{uX,n}, \ell_{uu,n}$
        - 終端コスト $\phi_{X}, \phi_{XX}$

以上を設定した後、Pythonによる実装を行う。

## 制御モデル 非線形バネ

下記のバネ-マスモデルでバネ$F(x)$を非線形なモデルとして取り扱い、iLQRでどのように動かすのかを学習する。

<p align="center">
<img src="./images/nonlinear_spring.png" style="width:35%;" />
</p>


$x(t)$に対する運動方程式は次である。

$$
m \ddot{x}(t) = u(t) - k x(t) - k_3 x(t)^3
$$

$v(t) = \dot{x}(t)$とし、状態$X=[x(t) , v(t)]^T$とすると、非線形状態方程式は以下となる。

$$
\begin{bmatrix}
\dot{x}(t) \\ \dot{v}(t)
\end{bmatrix} =
\begin{bmatrix}
v(t) \\
\frac{1}{m}\left( u(t) - k x(t) - k_3 x(t)^3 \right)
\end{bmatrix}
$$

前進オイラー法を用いて、上式は次の離散状態空間表現となる。$\Delta t$は離散時間である。

$$
\boxed{
\begin{bmatrix}
{x}_{n+1} \\ {v}_{n+1}
\end{bmatrix} =
\begin{bmatrix}
x_{n} + \Delta t v_{n} \\
v_{n} + \frac{\Delta t}{m}\left( u_n - k x_n - k_3 x_n^3 \right)
\end{bmatrix}
}
$$

上式は非線形であり$X_{n+1} = f(X_n , u_n)$として表す。




## コスト関数

総コスト$J$を次の構成とする。今回はステージコストは時刻によって変動させず、一定とする。

$$
\boxed{
J(X,U) = \phi(X_N) + \sum_{n=0}^{N-1} \ell(X_n, u_n)
}
$$

状態誤差を次のように設定する。

$$
e_x = x - x_{ref} , \quad e_v = v - v_{ref}
$$

終端では以下となる。

$$
e_{x,N} = x_N - x_{ref} , \quad e_{v,N} = v_N - v_{ref}
$$

目標位置$x_{ref}$において、速度と加速度をゼロに保つために必要な平衡入力を求める。運動方程式に$x=x_{ref}, \dot{x} = \ddot{x} = 0$を代入すると、制御入力は以下となる。

$$
u_{ref} = k \ x_{ref} + k_3 x_{ref}^3
$$ 

そして、入力誤差を次のように設定する。

$$
e_u = u - u_{ref}
$$

### ステージコスト $\ell$

コスト関数を基準軌道の周辺で二次近似する過程を確認できるように、位置誤差の4次項を含むステージコストを設定する。

$$
\boxed{
\ell(X, u) = \frac{1}{2} q_x e_x^2 + \frac{1}{2} q_v e_v^2 + \frac{1}{2} r e_u^2 + \frac{1}{4} q_4 e_x^4
}
$$

$q_x, q_v, r, q_4$はそれぞれの誤差に対する重みであり、スカラーである。

各項の役割は次となる。

- $e_x^2$ : 位置誤差を小さくする
- $e_v^2$ : 速度誤差を小さくする
- $e_u^2$ : 入力誤差を小さくする
- $e_x^4$ : 位置誤差が大きく離れた状態を強く罰する

### 終端コスト $\phi$

コスト関数を基準軌道の周辺で二次近似する過程を確認できるように、位置誤差の4次項を含む終端コストを設定する。

$$
\boxed{
\phi(X_N) = \frac{1}{2} q_{x,N} e_{x,N}^2 + \frac{1}{2} q_{v,N} e_{v,N}^2 + \frac{1}{4} q_{4,N} e_{x,N}^4
}
$$

$q_{x,N}, q_{v,N}, q_{4,N}$は終端時刻での各誤差に対する重みであり、スカラーである。

- $e_{x,N}^2$ : 終端時刻の位置誤差を小さくする
- $e_{v,N}^2$ : 終端時刻の速度誤差を小さくする
- $e_{x,N}^4$ : 終端時刻の位置誤差が大きく離れた状態を強く罰する


## モデルの一次近似と、コスト関数の2次近似

#### 非線形モデルの一次近似

以下の非線形バネモデル$X_{n+1} = f(X_n , u_n)$を$X_n$と$u_n$で偏微分する。

$$
\begin{bmatrix}
{x}_{n+1} \\ {v}_{n+1}
\end{bmatrix} =
\begin{bmatrix}
x_{n} + \Delta t v_{n} \\
v_{n} + \frac{\Delta t}{m}\left( u_n - k x_n - k_3 x_n^3 \right)
\end{bmatrix}
$$

$$
\begin{aligned}
\frac{\partial f(X_n, u_n)}{\partial X_n} &= \begin{bmatrix}
\dfrac{\partial x_{n+1}}{\partial x_n} & \dfrac{\partial x_{n+1}}{\partial v_n} \\[8pt]
\dfrac{\partial v_{n+1}}{\partial x_n} & \dfrac{\partial v_{n+1}}{\partial v_n} \\[8pt]
\end{bmatrix} 
= \begin{bmatrix}
1  & \Delta t \\[8pt]
-\dfrac{\Delta t}{m}(k + 3 k_3 x_n^2) & 1
\end{bmatrix} \\
\frac{\partial f(X_n, u_n)}{\partial u_n} &= \begin{bmatrix}
\dfrac{\partial x_{n+1}}{\partial u_n}  \\[8pt]
\dfrac{\partial v_{n+1}}{\partial u_n} 
\end{bmatrix} 
= \begin{bmatrix}
0 \\[8pt]
\dfrac{\Delta t}{m}
\end{bmatrix}
\end{aligned}
$$

ここから、一次近似摂動モデル$\delta X_{n+1}$を以下のように構成する。

$$
\delta X_{n+1} \approx A_n \delta X_n + B_n \delta u_n
$$

$$
\boxed{
A_n = \left. \frac{\partial f(X_n, u_n)}{\partial X_n} \right|_{(\bar{X}_n, \bar{u}_n)} = \begin{bmatrix}
1  & \Delta t \\[8pt]
-\dfrac{\Delta t}{m}(k + 3 k_3 \bar{x}_n^2) & 1
\end{bmatrix}}
$$

$$
\boxed{
B_n = \left. \frac{\partial f(X_n, u_n)}{\partial u_n} \right|_{(\bar{X}_n, \bar{u}_n)} =\begin{bmatrix}
0 \\[8pt]
\dfrac{\Delta t}{m}
\end{bmatrix}}
$$


#### コスト関数の二次近似

ステージコスト $\ell$と終端コスト$\phi$をそれぞれ二次近似するが、必要なのは、Taylor展開の1次項と2次項の係数である。

$$
\ell(X, u) = \frac{1}{2} q_x e_x^2 + \frac{1}{2} q_v e_v^2 + \frac{1}{2} r e_u^2 + \frac{1}{4} q_4 e_x^4
$$

$$
\phi(X_N) = \frac{1}{2} q_{x,N} e_{x,N}^2 + \frac{1}{2} q_{v,N} e_{v,N}^2 + \frac{1}{4} q_{4,N} e_{x,N}^4
$$

$$
e_x = x - x_{ref} , \quad e_v = v - v_{ref}, \quad e_u = u - u_{ref}
$$

$$
e_{x,N} = x_N - x_{ref} , \quad e_{v,N} = v_N - v_{ref}
$$

##### ステージコストの２次近似係数

$$
\begin{aligned}
\ell_{X,n} &=
\left. \frac{\partial \ell(X,u)}{\partial X} \right|_{(\bar{X}_n, \bar{u}_n)}=
\left. \begin{bmatrix} 
\dfrac{\partial \ell(X,u)}{\partial x} \\[8pt]
\dfrac{\partial \ell(X,u)}{\partial v}
\end{bmatrix}  \right|_{(\bar{X}_n, \bar{u}_n)} =
\left. \begin{bmatrix}
q_x e_x + q_4 e_x^3 \\
q_v e_v
\end{bmatrix}  \right|_{(\bar{X}_n, \bar{u}_n)} =
\begin{bmatrix}
q_x (\bar{x}_n - x_{ref}) + q_4 (\bar{x}_n - x_{ref})^3 \\
q_v (\bar{v}_n - v_{ref})
\end{bmatrix} \\
\ell_{u,n} &= \left. \frac{\partial \ell(X,u)}{\partial u} \right|_{(\bar{X}_n, \bar{u}_n)} = \left. r e_u \right|_{(\bar{X}_n, \bar{u}_n)} = r (\bar{u}_n - u_{ref}) \\
\ell_{XX,n} &= \left. \frac{\partial^2 \ell(X,u)}{\partial X^2} \right|_{(\bar{X}_n, \bar{u}_n)}=
\left. \begin{bmatrix} 
\dfrac{\partial^2 \ell(X,u)}{\partial x^2} & \dfrac{\partial^2 \ell(X,u)}{\partial x \partial v} \\[8pt]
\dfrac{\partial^2 \ell(X,u)}{\partial v \partial x} & \dfrac{\partial^2 \ell(X,u)}{\partial v^2}
\end{bmatrix} \right|_{(\bar{X}_n, \bar{u}_n)} = 
\left. \begin{bmatrix}
q_x + 3 q_4 e_x^2 & 0 \\
0 & q_v
\end{bmatrix} \right|_{(\bar{X}_n, \bar{u}_n)} =
\begin{bmatrix}
q_x + 3 q_4 (\bar{x}_n - x_{ref})^2 & 0 \\
0 & q_v
\end{bmatrix} \\
\ell_{uX,n} &= \left. \frac{\partial}{\partial X} \frac{\partial \ell(X,u)}{\partial u} \right|_{(\bar{X}_n, \bar{u}_n)}= 
\left. \begin{bmatrix} 
\dfrac{\partial^2 \ell(X,u)}{\partial u \partial x} & \dfrac{\partial^2 \ell(X,u)}{\partial u\partial v}
\end{bmatrix}  \right|_{(\bar{X}_n, \bar{u}_n)} = \begin{bmatrix} 0 & 0 \end{bmatrix} \\
\ell_{uu,n} &= \left. \frac{\partial^2 \ell(X,u)}{\partial u^2} \right|_{(\bar{X}_n, \bar{u}_n)} = r
\end{aligned}
$$

まとめると以下となる。

$$
\boxed{
\begin{aligned}
\ell_{X,n} &= \begin{bmatrix}
q_x (\bar{x}_n - x_{ref}) + q_4 (\bar{x}_n - x_{ref})^3 \\
q_v (\bar{v}_n - v_{ref})
\end{bmatrix} \\
\ell_{u,n} &= r (\bar{u}_n - u_{ref}) \\
\ell_{XX,n} &= \begin{bmatrix}
q_x + 3 q_4 (\bar{x}_n - x_{ref})^2 & 0 \\
0 & q_v
\end{bmatrix} \\
\ell_{uX,n} &= \begin{bmatrix} 0 & 0 \end{bmatrix} \\
\ell_{uu,n} &= r
\end{aligned}
}
$$






#### 終端コストの2次近似


$$
\begin{aligned}
\phi_X &= \left. \frac{\partial \phi}{\partial X_N} \right|_{\bar{X}_N} = 
\left. \begin{bmatrix}
\dfrac{\partial \phi}{\partial x_N} \\ \dfrac{\partial \phi}{\partial v_N}
\end{bmatrix}  \right|_{\bar{X}_N} =
\left. \begin{bmatrix}
q_{x,N} e_{x,N} + q_{4,N} e_{x,N}^3 \\ q_{v,N} e_{v,N}
\end{bmatrix}  \right|_{\bar{X}_N} =
\begin{bmatrix}
q_{x,N} (\bar{x}_N - x_{ref}) + q_{4,N} (\bar{x}_N - x_{ref})^3 \\ q_{v,N} (\bar{v}_N - v_{ref})
\end{bmatrix} \\
\phi_{XX} &= \left. \frac{\partial^2 \phi}{\partial X_N^2} \right|_{\bar{X}_N} = 
\left. \begin{bmatrix}
\dfrac{\partial^2 \phi}{\partial x_N^2} & \dfrac{\partial^2 \phi}{\partial v_N \partial x_N }\\
\dfrac{\partial^2 \phi}{\partial x_N \partial v_N} & \dfrac{\partial^2 \phi}{\partial v_N^2}
\end{bmatrix}  \right|_{\bar{X}_N} =
\left. \begin{bmatrix}
q_{x,N} + 3 q_{4,N} e_{x,N}^2 & 0 \\
0 & q_{v,N}
\end{bmatrix}  \right|_{\bar{X}_N} =
\begin{bmatrix}
q_{x,N} + 3 q_{4,N} (\bar{x}_N - x_{ref})^2 & 0 \\
0 & q_{v,N}
\end{bmatrix}
\end{aligned}
$$

まとめると以下となる。

$$
\boxed{
\begin{aligned}
\phi_X &= \begin{bmatrix}
q_{x,N} (\bar{x}_N - x_{ref}) + q_{4,N} (\bar{x}_N - x_{ref})^3 \\ q_{v,N} (\bar{v}_N - v_{ref})
\end{bmatrix} \\
\phi_{XX} &=
\begin{bmatrix}
q_{x,N} + 3 q_{4,N} (\bar{x}_N - x_{ref})^2 & 0 \\
0 & q_{v,N}
\end{bmatrix}
\end{aligned}
}
$$





## Python による実装

In [2]:
import numpy as np
from scipy.linalg import cho_factor, cho_solve
from numba import jit
from typing import NamedTuple

class Parameters(NamedTuple):
    dt: float # 有限ホライゾンの離散時間

    # モデル
    k: float = 0.5 
    k3: float = 1.5
    m: float = 0.1

    # ステージコスト重み
    qx: float = 1
    qv: float = 2
    r: float = 1
    q4: float  = 1

    # 終端コスト重み
    qxN: float = 10
    qvN: float = 10
    q4N: float = 5

# 非線形

# 非線形状態空間表現
@jit(cache=True)
def calculate_f(X,u, 
                params:Parameters):
    x = X[0]
    v = X[1]
    x_next = x + params.dt * v
    v_next = v + params.dt/params.m*(u - params.k * x - params.k3 * x**3)

    return np.array([x_next, v_next])

# ステージコストの計算
@jit(cache=True)
def calculate_stage_cost(X, u,
                         x_ref, v_ref, u_ref,
                         params:Parameters):
    x = X[0]
    v = X[1]
    ex = x - x_ref
    ev = v - v_ref
    eu = u - u_ref
    return 0.5*(params.qx*ex**2 + params.qv*ev**2 + params.r*eu**2) + 0.25 *params.q4*ex**4

# 終端コストの計算
@jit(cache=True)
def calculate_terminal_cost(X,
                            x_ref, v_ref,
                            params:Parameters):
    x = X[0]
    v = X[1]
    ex = x - x_ref
    ev = v - v_ref
    return 0.5*(params.qxN*ex**2 + params.qvN*ev**2) + 0.25*params.q4N*ex**4

# 総コスト関数の計算
@jit(cache=True)
def calculate_J(X:np.ndarray, U:np.ndarray,
                x_ref, v_ref, u_ref,
                params:Parameters):
    N = U.shape[0]

    stage_cost = 0
    for n in range(N):
        stage_cost += calculate_stage_cost(X[n], U[n],
                                    x_ref, v_ref, u_ref,
                                    params)

    terminal_cost = calculate_terminal_cost(X[N],
                                  x_ref, v_ref,
                                  params)

    return stage_cost + terminal_cost

# 一次、二次近似の係数

@jit(cache=True)
def calculate_A(X, 
                params:Parameters):
    x = X[0]
    return np.array([
        [1 , params.dt],
        [-params.dt/params.m*(params.k + 3*params.k3*x**2), 1]
    ])

@jit(cache=True)
def calculate_B(params:Parameters):
    return np.array([
        [0.0],
        [params.dt/params.m]
    ])


# ステージコスト
@jit(cache=True)
def calculate_s_x(X,
                  x_ref, v_ref,
                  params:Parameters):
    x = X[0]
    v = X[1]
    ex = x - x_ref
    ev = v - v_ref
    return np.array([
        [params.qx*ex + params.q4*ex**3],
        [params.qv*ev]
    ])

@jit(cache=True)
def calculate_s_u(u, u_ref, params:Parameters):
    return params.r*(u - u_ref)

@jit(cache=True)
def calculate_s_XX(X, x_ref,
                   params:Parameters):
    x = X[0]
    return np.array([
        [params.qx + 3*params.q4*(x-x_ref)**2, 0],
        [0, params.qv]
    ])

@jit(cache=True)
def calculate_s_uX():
    return np.zeros((1,2))

def calculate_s_uu(params:Parameters):
    return params.r

@jit(cache=True)
def calculate_t_X(X,
                  x_ref, v_ref,
                  params:Parameters):
    x = X[0]
    v = X[1]
    ex = x - x_ref
    ev = v - v_ref
    return np.array([
        [params.qxN*ex + params.q4N*ex**3],
        [params.qvN*ev]        
    ])

@jit(cache=True)
def calculate_t_XX(X, x_ref,
                   params:Parameters):
    x = X[0]
    return np.array([
        [params.qxN + 3*params.q4N*(x - x_ref)**2 , 0],
        [0 , params.qvN]
    ])

# forward pass
@jit(cache=True)
def calculate_forward_pass(X_array:np.ndarray, U_array:np.ndarray,
                           k_array:np.ndarray, K_array:np.ndarray,
                           params:Parameters,
                           alpha=1.0,
                           ):

    N = U_array.shape[0]
    nx = X_array.shape[1]
    nu = U_array.shape[1]
    X_cand_array = np.zeros((N+1, nx))
    U_cand_array = np.zeros((N, nu))

    X_cand_array[0] = X_array[0].copy()

    for n in range(N):
        dX_next = X_cand_array[n] - X_array[n]

        u_next = U_array[n] + alpha * k_array[n] + K_array[n] @ dX_next

        X_cand_array[n+1] = calculate_f(X_cand_array[n], u_next, params)
        U_cand_array[n] = u_next

    return X_cand_array, U_cand_array

@jit(cache=True)
def line_search_with_forward_pass(X_array:np.ndarray, U_array:np.ndarray,
                                  k_array:np.ndarray, K_array:np.ndarray,
                                  params:Parameters,
                                  x_ref, v_ref, u_ref,
                                  alpha=1.0,
                                  redunction_rate = 0.5,
                                  iteration_max=10):

    alpha = alpha

    bar_J = calculate_J(X_array, U_array,
                        x_ref, v_ref, u_ref,
                        params)

    for _ in range(iteration_max):
        X_cand_array, U_cand_array = calculate_forward_pass(
            X_array,U_array,
            k_array, K_array,
            params,
            alpha
        )

        cand_J = calculate_J(X_cand_array, U_cand_array,
                             x_ref, v_ref, u_ref,
                             params)

        if cand_J < bar_J:
            return X_cand_array, U_cand_array, alpha
        else:
            alpha*=redunction_rate
 
    raise ValueError("Could not find a optimized trajectory")


@jit(cache=True)
def backward_pass(X_array:np.ndarray, U_array:np.ndarray,
                  x_ref, v_ref, u_ref,
                  params:Parameters):
    N = U_array.shape[0]
    nx = X_array.shape[1]
    nu = U_array.shape[1]

    k_array = np.zeros((N, nu))
    K_array = np.zeros((N, nu, nx))

    # 終端価値関数
    Vx = calculate_t_X(X_array[N], x_ref, v_ref, params)
    Vxx = calculate_t_XX(X_array, x_ref, params)
    Vxx = 0.5 *(Vxx + Vxx.T) # 対称化

    for n in range(N-1, -1, -1):
        # ステージコストの係数
        ell_x = calculate_s_x(X_array[n], x_ref, v_ref, params)
        ell_u = calculate_s_u(U_array[n], u_ref)
        ell_uu = calculate_s_uu(params)
        ell_uX = calculate_s_uX()
        ell_xx = calculate_s_XX(X_array[n], x_ref, params)
        # 一次近似摂動動力学の係数
        A = calculate_A(X_array[n], params)
        B = calculate_B(params)
        # Q関数の二次近似の係数
        Qx = ell_x + A.T@Vx
        Qu = ell_u + B.T@Vx
        Quu = ell_uu + B.T@Vxx@B
        Qux = ell_uX + B.T@Vxx@A
        Qxx = ell_xx + A.T@Vxx@A

        # 対称化
        Quu = 0.5*(Quu + Quu.T)

        # 入力修正項の係数
        try:
            factor = cho_factor(Quu, lower=True)
        except np.linalg.LinAlgError:
            raise(
                f"Quu is not positive definite at n={n}"
            )

        kn = cho_solve(factor,-Qu)
        Kn = cho_solve(factor, -Qux)

        k_array[n] = kn
        K_array[n] = Kn

        # 価値関数を時刻nへ更新
        Vx = Qx + Qux.T@kn
        Vxx = Qxx + Qux.T@Kn

        #対称化
        Vxx = 0.5*(Vxx + Vxx.T)

    return k_array, K_array

@jit(cache=True)
def calculate_rollout(X, U_array, params:Parameters):

    N = U_array.shape[0]
    nx = X.shape[0]
    X_array = np.zeros((N, nx))
    X_array[0] = X
    for n in range(N):
        X_array[n+1] = calculate_f(X_array[n], U_array[n], params)

    return X_array    

def calulate_iLQR(X, U_array, 
                  x_ref, v_ref, u_ref,
                  params:Parameters, iteration_max=30,
                  eps=1e-4):

    # rolloutし、基準軌道を計算
    base_X = calculate_rollout(X, U_array, params)
    base_U = U_array.copy()

    base_J = calculate_J(base_X, base_U, x_ref, v_ref, u_ref, params)

    # 記録用
    alpha_array = []

    for i in range(iteration_max):
        # backward pass で入力修正則を求める
        k_array, K_array = backward_pass(base_X, base_U, x_ref, v_ref, u_ref, params)

        # forward pass で今回の反復における基準軌道を計算
        cand_X, cand_U, alpha = line_search_with_forward_pass(base_X, base_U, k_array, K_array, params,
                                                              x_ref, v_ref, u_ref)

        # 収束の判断
        next_J = calculate_J(cand_X, cand_U, x_ref, v_ref, u_ref, params)
        if np.abs(base_J - next_J) < eps:
            print(f"iLQR find optimized trajectory at iteration {i}")
            return cand_X, cand_U, alpha_array
        else:

            base_X = cand_X
            base_U = cand_U
            alpha_array.append(alpha)

    raise ValueError(f"iLQR could not find optimezed trajector of this parameters.")
        
